# 03. Power Quality Compliance: Voltage Regulation & IEEE Std 141 Phase Imbalance

## Electrical Engineering Context
Power Quality directly impacts equipment lifespan, motor torque stability, and technical efficiency. In this notebook, we evaluate:
1. **Voltage Regulation (IEEE Std 1159 / IEC 60038):**
   $$\text{Voltage Deviation \%} = \frac{V_{actual} - V_{nominal}}{V_{nominal}} \times 100$$
   Nominal limits are $\pm 5\%$. Sags $<-10\%$ risk motor stalls and contactor dropouts.
2. **Current Imbalance (IEEE Std 141 / NEMA MG 1):**
   $$I_{imbalance} = \frac{\max(|I_r - I_{avg}|, |I_y - I_{avg}|, |I_b - I_{avg}|)}{I_{avg}} \times 100$$
   An imbalance of $5\%$ induces a $25\%$ increase in motor winding temperature rise!
3. **Power Factor & Reactive Energy Penalties:**
   Operational compliance below $0.85$ triggers regulatory financial penalties.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.utils.config import PROCESSED_DATA_DIR

df_meas = pd.read_csv(PROCESSED_DATA_DIR / "fact_electrical_measurements.csv")
dim_fdr = pd.read_csv(PROCESSED_DATA_DIR / "dim_feeder.csv")


## 1. Voltage Deviation vs Regulatory Standard Bands
We categorize intervals into Normal $(\pm 5\%)$, Overvoltage $(>+5\%)$, Undervoltage $(-5\% \text{ to } -10\%)$, and Critical Sags $(<-10\%)$.


In [ ]:
v_band_counts = df_meas["voltage_quality_band"].value_counts()
print("Voltage Quality Compliance Summary:")
print(v_band_counts)

plt.figure(figsize=(8, 4.5))
sns.countplot(data=df_meas, y="voltage_quality_band", palette="coolwarm", order=v_band_counts.index)
plt.title("Distribution of IEEE 1159 Voltage Deviation Bands", fontsize=12, fontweight="bold")
plt.xlabel("Telemetry Measurement Count")
plt.tight_layout()
plt.show()


## 2. Phase Current Imbalance Across Feeders (IEEE Std 141)
We identify feeders with chronic unbalance exceeding the $5\%$ warning and $10\%$ critical limits.


In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_meas, x="feeder_id", y="current_imbalance_pct", palette="magma")
plt.axhline(5.0, color="orange", linestyle="--", label="IEEE 141 5% Warning Threshold")
plt.axhline(10.0, color="red", linestyle="-", label="10% Critical Imbalance Limit")
plt.title("Phase Current Imbalance Distribution by Feeder", fontsize=12, fontweight="bold")
plt.xlabel("Feeder ID")
plt.ylabel("Current Imbalance %")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Power Factor Tariff Penalty Exposure
We calculate the cumulative hours each feeder operated below $0.85$ power factor.


In [ ]:
pf_pen = df_meas.groupby("feeder_id")["pf_penalty_incurred"].agg(
    intervals="sum",
    penalty_hours=lambda s: s.sum() * 0.5,
    pct_time=lambda s: (s.sum() / len(s)) * 100.0
).reset_index().sort_values(by="penalty_hours", ascending=False)

print("Top Feeders by Commercial PF Penalty Exposure:")
print(pf_pen.head(6))
